# Week 4 — Nonlinear Methods

FIN 43800 · Machine Learning in Finance

Andreas Neuhierl

# Where We Are

## Where Trees Fit

- **Weeks 1–3:** out of sample is the test. Many predictors, few months:
  shrink the slopes, and tune how hard without peeking.
- **Every model so far was a straight line:** each predictor moves the
  forecast by its slope, whatever the others are.
- **Week 4: curves and interactions.**
  - Real returns have an interaction: value pays among small firms.
  - **One tree** asks yes-or-no questions and forecasts an average.
  - **Boosting** adds many small trees; the focus of the week.
  - Where trees win, and where they cannot: enough data, or not.

## Upload, Then Run Setup

Starting a new Colab session? Colab deletes uploaded files when a
session ends. Download `size_value_monthly.csv` and `gw_monthly.csv`
from Brightspace, drag them into the Files pane (folder icon on the
left), then run the **Setup** cell and the **helpers** cell below before
any other cell.

## Setup: What It Builds

- **`portfolios`**: monthly excess returns of 25 portfolios, July 1963
  to December 2025. Stocks sorted five ways by **size** (market value)
  and five ways by **book-to-market** (B/M): accounting value over
  market value. High B/M is “value” (cheap), low is “growth”.
- **`monthly`**: the Goyal–Welch file of week 2.
- Source of the first: Kenneth French’s Data Library.

<!-- -->

- **Your Turn** code is text in this notebook: click **+ Code**, paste
  it, fill the blank, then Shift + Enter.

Run this first, every session.

In [ ]:
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Fetch the one helper file we need from the course repo.
    import subprocess
    subprocess.run(
        ["wget", "-q", "-O", "mlfin_data.py",
         "https://raw.githubusercontent.com/aneuhierl/ml-in-finance-fall2026/main/src/mlfin_data.py"],
        check=True,
    )
else:
    # Find src/mlfin_data.py by walking UP from this notebook's folder.
    here = Path.cwd().resolve()
    src_folders = [folder / "src" for folder in [here, *here.parents]]
    matches = [folder for folder in src_folders if (folder / "mlfin_data.py").is_file()]
    sys.path.insert(0, str(matches[0]) if matches else "../src")
from mlfin_data import resolve_data_path

import numpy as np                 # arrays of numbers
import pandas as pd                # tables with columns
import matplotlib.pyplot as plt    # charts
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPRegressor

# Kenneth French's 25 portfolios formed on size and book-to-market (value-weighted), per
# month, July 1963 to December 2025.
# Data: Copyright Eugene F. Fama and Kenneth R. French (French's Data Library).
# Returns are SIMPLE, in decimals per month (0.01 = 1%). rf: the T-bill return of the same
# month. Columns ME1_BM1 ... ME5_BM5: ME is size (1 = the smallest fifth of firms, 5 = the
# largest); BM is book-to-market (1 = the lowest fifth, "growth"; 5 = the highest, "value").
# They are RAW returns, so we subtract rf to get EXCESS returns.
french = pd.read_csv(resolve_data_path("size_value_monthly.csv"))
french.index = pd.to_datetime(french["yyyymm"].astype(str), format="%Y%m")
portfolio_names = [f"ME{size}_BM{bm}" for size in range(1, 6) for bm in range(1, 6)]
portfolios = french[portfolio_names].sub(french["rf"], axis=0)      # EXCESS returns

# The Goyal-Welch monthly file of week 2. equity = stocks minus T-bills, simple, in decimals.
monthly = pd.read_csv(resolve_data_path("gw_monthly.csv"))
monthly.index = pd.to_datetime(monthly["yyyymm"].astype(str), format="%Y%m")
plt.rcParams["font.size"] = 18     # big text for the projector
print(portfolios.shape, monthly.shape)

Then run this cell too. It defines two helpers that draw a fitted tree:
as a flowchart, and as shaded leaves on a map of two characteristics.
You need not read them.

In [ ]:
def describe_question(tree, node):
    """The question a fitted tree asks at one node, in words, such as 'size <= 0.03?'."""
    name = tree.feature_names_in_[tree.tree_.feature[node]]
    cut = tree.tree_.threshold[node]
    if cut % 1 in (0, 0.5):                  # a cut between whole numbers: 1, 2, ..., 5
        return f"{name} ≤ {int(cut)}?"
    return f"{name} ≤ {cut:.2f}?".replace("-", "−")


def draw_tree(tree, axis, leaf_text=None):
    """Draw a fitted tree of depth 2 as a flowchart: yes goes left, no goes right.
    Each leaf shows its average forecast, unless leaf_text(number, node) gives other text."""
    nodes = tree.tree_
    assert nodes.node_count == 7, "draw_tree draws trees of depth 2 only"
    # scikit-learn numbers a depth-2 tree's nodes top down, left first:
    # 0 = the first question, 1 and 4 = the two questions below it, 2, 3, 5, 6 = the leaves.
    spots = {0: (0.5, 0.92), 1: (0.25, 0.55), 4: (0.75, 0.55),
             2: (0.12, 0.09), 3: (0.38, 0.09), 5: (0.62, 0.09), 6: (0.88, 0.09)}
    for parent, left, right in [(0, 1, 4), (1, 2, 3), (4, 5, 6)]:
        for child, word, side in [(left, "yes", "right"), (right, "no", "left")]:
            to_a_leaf = child in (2, 3, 5, 6)
            axis.annotate("", xy=spots[child], xytext=spots[parent],
                          arrowprops=dict(arrowstyle="->", linewidth=1.5, shrinkA=11,
                                          shrinkB=20 if to_a_leaf else 11))
            middle_x = (spots[parent][0] + spots[child][0]) / 2
            middle_y = (spots[parent][1] + spots[child][1]) / 2
            axis.text(middle_x + (-0.02 if side == "right" else 0.02), middle_y, word,
                      ha=side, va="center", fontsize=14, color="gray")
    for node in (0, 1, 4):
        axis.text(*spots[node], describe_question(tree, node), ha="center", va="center",
                  fontsize=14, bbox=dict(boxstyle="round", facecolor="white", edgecolor="black"))
    for number, node in enumerate((2, 3, 5, 6), start=1):
        forecast = 100 * nodes.value[node][0][0]          # the leaf's average, in percent
        text = f"leaf {number}\n{forecast:.2f}%".replace("-", "−")
        if leaf_text is not None:
            text = leaf_text(number, node)
        axis.text(*spots[node], text, ha="center", va="center", fontsize=14,
                  bbox=dict(boxstyle="square", facecolor="#eeeeee", edgecolor="gray"))
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.set_axis_off()


def shade_leaves(tree, axis, x_name, x_range, y_name, y_range, colors, limits):
    """Shade the map of two characteristics by the tree's forecast, one color per leaf,
    and write each leaf's number in it."""
    x_grid, y_grid = np.meshgrid(np.linspace(*x_range, 300), np.linspace(*y_range, 300))
    grid = pd.DataFrame({x_name: x_grid.ravel(), y_name: y_grid.ravel()})
    grid = grid[list(tree.feature_names_in_)]             # the columns in the tree's order
    forecasts = 100 * tree.predict(grid).reshape(x_grid.shape)
    leaves = tree.apply(grid).reshape(x_grid.shape)        # the leaf each point lands in
    axis.pcolormesh(x_grid, y_grid, forecasts, cmap=colors, vmin=limits[0], vmax=limits[1],
                    shading="auto")
    for number, leaf in enumerate(np.unique(leaves), start=1):
        inside = leaves == leaf
        axis.text(x_grid[inside].mean(), y_grid[inside].mean(), f"leaf {number}",
                  ha="center", va="center", fontsize=15,
                  bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    axis.set_xlabel(x_name)
    axis.set_ylabel(y_name)

# Curves and Interactions

## Many Assets, One Month

- **Weeks 2 and 3:** one series through time. Predict the market’s next
  month from a few time-series predictors.
- **This week: the cross-section.** Many assets at once. Predict which
  ones earn **more than the others** from their characteristics.
- **One row** is one portfolio (later, one stock) in one month.
  - **Inputs:** its characteristics at the start of the month, such as
    size and book-to-market.
  - **Target:** its return that month.
- Same rules as before: judge every model on data it has not seen.

## Straight Lines Add Up

- Every model so far is a straight line. In size and value:
  $$\text{forecast} = a + b \times \text{size} + c \times \text{value}$$
- Value moves the forecast by $c$, **whatever the size**: if value adds
  0.2% a month for big firms, the line says it adds 0.2% for small firms
  too.
- **Interaction:** the effect of one predictor depends on another.

## Value Pays Among Small Firms

In [ ]:
# Average monthly excess return of each portfolio, July 1963 to December 2025, in percent.
# Rows: size group 1 (smallest firms) to 5 (largest). Columns: B/M group 1 (growth) to 5 (value).
averages = pd.DataFrame(100 * portfolios.mean().to_numpy().reshape(5, 5),
                        index=[f"size {size}" for size in range(1, 6)],
                        columns=[f"B/M {bm}" for bm in range(1, 6)])
value_spread = averages["B/M 5"] - averages["B/M 1"]     # value minus growth, per size group

figure, axis = plt.subplots(figsize=(10, 2.8))
axis.imshow(averages, cmap="Blues", vmin=0, vmax=1.7, aspect="auto")
for row in range(5):
    for column in range(5):
        axis.text(column, row, f"{averages.iloc[row, column]:.2f}", ha="center", va="center")
    axis.text(4.75, row, f"   value spread {value_spread.iloc[row]:.2f}", va="center")
axis.set_xlim(-0.5, 7.4)
axis.set_xticks(range(5), ["1 growth", "2", "3", "4", "5 value"])
axis.set_yticks(range(5), ["1 small", "2", "3", "4", "5 big"])
axis.set_xlabel("book-to-market group")
axis.set_ylabel("size group")
for side in ["top", "right"]:
    axis.spines[side].set_visible(False)
plt.show()

- Average monthly excess return, %. The value spread is several times
  larger among the smallest firms than among the largest.
- How much value pays **depends on size**: an interaction.

## A Straight Line Misses the Corner

In [ ]:
# A straight-line model of the 25 averages: average = a + b * size group + c * B/M group.
groups = pd.DataFrame([(size, bm) for size in range(1, 6) for bm in range(1, 6)],
                      columns=["size group", "B/M group"])
line = LinearRegression().fit(groups, averages.to_numpy().ravel())
fitted = line.predict(groups).reshape(5, 5)
misses = averages.to_numpy() - fitted            # actual minus the line, % per month

figure, axis = plt.subplots(figsize=(10, 2.5))
axis.imshow(misses, cmap="RdBu", vmin=-0.7, vmax=0.7, aspect="auto")
for row in range(5):
    for column in range(5):
        axis.text(column, row, f"{misses[row, column]:+.2f}".replace("-", "−"),
                  ha="center", va="center")
axis.set_xticks(range(5), ["1 growth", "2", "3", "4", "5 value"])
axis.set_yticks(range(5), ["1 small", "2", "3", "4", "5 big"])
axis.set_xlabel("book-to-market group")
axis.set_ylabel("size group")
plt.show()

- **The linear model:** average
  $= a + b\cdot\text{size} + c\cdot\text{B/M}$, with the group numbers 1
  to 5. Least squares picks the $a$, $b$, $c$ that fit the 25 averages
  best.
- **Fitted** is the formula at a portfolio’s two group numbers; the
  **miss** is actual minus fitted, % a month (shown). Each B/M step adds
  $c$ points, the same in every size group.
- Its biggest miss is the small-growth corner.

## Interactions by Hand Explode

In [ ]:
from math import comb      # comb(n, k): the number of ways to choose k things out of n
for count in [12, 40, 153]:
    print(f"{count} predictors: {comb(count, 2):,} pairs, {comb(count, 3):,} triples")

- You could add products by hand: size × B/M. But **which** of them?
  Week 2 had 12 predictors; a stock-level panel built from Jensen, Kelly
  & Pedersen’s (2023) data has 40, and their full set has 153.
- And **which shape**: a product? “only the smallest fifth”? A tree
  searches over both.

# One Tree

## A Tree Is a Flowchart

In [ ]:
# Stack the 25 portfolios into one long table: one row per portfolio per month.
rows = []
for size in range(1, 6):
    for bm in range(1, 6):
        rows.append(pd.DataFrame({"size group": size, "B/M group": bm,
                                  "excess return": portfolios[f"ME{size}_BM{bm}"]}))
panel = pd.concat(rows)                                    # 25 x 750 = 18,750 rows

# A tree of depth 2: one question, then one more question on each side.
first_tree = DecisionTreeRegressor(max_depth=2)
first_tree.fit(panel[["size group", "B/M group"]], panel["excess return"])

figure, axes = plt.subplots(1, 2, figsize=(11, 3.5), gridspec_kw={"width_ratios": [1.5, 1]})
draw_tree(first_tree, axes[0])
shade_leaves(first_tree, axes[1], "B/M group", (0.5, 5.5), "size group", (0.5, 5.5),
            colors="Blues", limits=(0, 1.4))
for edge in [1.5, 2.5, 3.5, 4.5]:                          # the 25 portfolios, as cells
    axes[1].axvline(edge, color="white", linewidth=1)
    axes[1].axhline(edge, color="white", linewidth=1)
axes[1].set_xticks(range(1, 6))
axes[1].set_yticks(range(1, 6))
axes[1].invert_yaxis()                                     # small firms on top, as before
plt.show()

- Back to returns, in the cross-section: one row is one portfolio in one
  month, with its size and B/M group as inputs.
- Each row answers the questions (yes left, no right) and lands in one
  **leaf**, an end point of the flowchart. **Depth 2:** two questions
  from the top to any leaf.
- A leaf forecasts the average of its months.

## A Split by Hand

In [ ]:
# How "explains" is worked out, on one row: the smallest firms' five averages, B/M group 1
# (growth) to 5 (value), % a month, rounded to two decimals as you would by hand. A question
# splits the row in two sides, and each side is forecast by its own average.
small_row = averages.loc["size 1"].round(2).to_numpy()


def squared_gaps(numbers):
    """Each number's gap from the numbers' own average, squared, then added up."""
    return np.sum((numbers - numbers.mean()) ** 2)


row_spread = squared_gaps(small_row)                 # the spread a question can explain
by_hand = {}
figure, axes = plt.subplots(1, 2, figsize=(11, 2.1), sharey=True)
for axis, cut in zip(axes, [1, 4]):                  # "B/M <= 1?" and "B/M <= 4?"
    yes, no = small_row[:cut], small_row[cut:]
    left_over = squared_gaps(yes) + squared_gaps(no)
    by_hand[cut] = {"yes": yes.mean(), "no": no.mean(), "left over": left_over,
                    "explains": 100 * (1 - left_over / row_spread)}
    axis.bar(range(1, 6), small_row, color="lightsteelblue")
    for position, value in zip(range(1, 6), small_row):
        axis.text(position, value + 0.04, f"{value:.2f}", ha="center", fontsize=14)
    for side, first, last in [(yes, 1, cut), (no, cut + 1, 5)]:   # each side's average
        axis.hlines(side.mean(), first - 0.45, last + 0.45, color="tab:orange", linewidth=4)
    axis.axvline(cut + 0.5, color="black", linestyle="--")
    axis.set_title(f"smallest firms: B/M ≤ {cut}?  explains {by_hand[cut]['explains']:.0f}%",
                   fontsize=16)
    axis.set_xticks(range(1, 6), ["1 growth", "2", "3", "4", "5 value"])
    axis.set_ylim(0, 1.35)
axes[0].set_ylabel("% a month")
plt.show()

- The smallest firms’ row, growth to value. **Spread:** each number’s
  squared gap from the row’s average, added up.
- **B/M ≤ 1?** Growth alone against the other four (orange: each side’s
  average). Add up the squared gaps left inside the two sides; the share
  of the spread that is gone is the share explained.
- **B/M ≤ 4?** leaves more of the spread over. In this row, cutting off
  growth wins.

## Three Candidate Questions

In [ ]:
# The 25 averages again. A question splits the 25 portfolios in two groups; each group is
# forecast by its own average. "Left over" is how far the 25 averages sit from their group's
# average: the smaller, the better the question. (With all 18,750 rows the ranking is the same:
# every portfolio has 750 months.)
table_25 = groups.copy()
table_25["average"] = averages.to_numpy().ravel()


def share_explained(column, cut):
    """Percent of the 25 averages' squared spread explained by the question 'column <= cut?'."""
    yes = table_25[table_25[column] <= cut]["average"]
    no = table_25[table_25[column] > cut]["average"]
    total = np.sum((table_25["average"] - table_25["average"].mean()) ** 2)
    left_over = np.sum((yes - yes.mean()) ** 2) + np.sum((no - no.mean()) ** 2)
    return 100 * (1 - left_over / total)


candidates = [("size group", 1), ("B/M group", 3), ("B/M group", 1)]
figure, axes = plt.subplots(1, 3, figsize=(11, 3.1))
for axis, (column, cut) in zip(axes, candidates):
    axis.imshow(averages, cmap="Blues", vmin=0, vmax=1.7, aspect="equal")
    if column == "B/M group":                     # a vertical cut: columns are B/M groups
        axis.axvline(cut - 0.5, color="tab:orange", linewidth=5)
        sides = [(cut / 2 - 0.5, 2, table_25[table_25[column] <= cut]["average"].mean()),
                 ((cut + 5) / 2 - 0.5, 2, table_25[table_25[column] > cut]["average"].mean())]
    else:                                         # a horizontal cut: rows are size groups
        axis.axhline(cut - 0.5, color="tab:orange", linewidth=5)
        sides = [(2, cut / 2 - 0.5, table_25[table_25[column] <= cut]["average"].mean()),
                 (2, (cut + 5) / 2 - 0.5, table_25[table_25[column] > cut]["average"].mean())]
    for x_position, y_position, group_average in sides:
        axis.text(x_position, y_position, f"{group_average:.2f}", ha="center", va="center",
                  fontsize=17, bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))
    axis.set_title(f"{column[:-6]} ≤ {cut}?  explains {share_explained(column, cut):.0f}%",
                   fontsize=17)
    axis.set_xticks([])
    axis.set_yticks([])
plt.show()

- Now all 25 averages, as a map (small firms on top, value on the
  right). Each side of the line is forecast by its own average.
- The best question leaves the **least** error.
- It explains less of all 25 than of the smallest firms’ row: growth
  trails the rest far among the smallest firms, and not at all among the
  largest.

## Trying Every Question

In [ ]:
questions, shares = [], []
for column in ["size group", "B/M group"]:
    for cut in [1, 2, 3, 4]:
        questions.append(f"{column} ≤ {cut}?")
        shares.append(share_explained(column, cut))
best = int(np.argmax(shares))
colors = ["tab:orange" if k == best else "gray" for k in range(len(shares))]
plt.figure(figsize=(10, 2.9))
plt.barh(questions, shares, color=colors)
plt.gca().invert_yaxis()
plt.xlabel("share of the spread of the 25 averages explained (%)")
plt.show()

- Try every characteristic and every cut; keep the question whose two
  averages explain the most. Then repeat inside each leaf.

## No Slopes, No Standardizing

- **No slopes:** a leaf forecasts the average of its training rows.
- **No standardizing:** “is size group ≤ 4?” gives the same leaves in
  any units (unlike ridge: “Standardize First”).
- Read: ISLP 8.1.

## Greedy: One Step Ahead

In [ ]:
# A made-up rule that needs two characteristics: the outcome is high when A and B are both above
# their middles or both below, and low when they disagree. 300 stocks, noise added.
def agree_sample(seed):
    """300 rows of two characteristics, A and B, between -1 and 1, and the outcome."""
    sample_generator = np.random.default_rng(seed)
    both = sample_generator.uniform(-1, 1, size=(300, 2))
    outcome = np.where(both[:, 0] * both[:, 1] > 0, 1.0, -1.0) + sample_generator.normal(0, 0.5, 300)
    return pd.DataFrame(both, columns=["A", "B"]), outcome


def fit_share(table, outcome, depth):
    """A tree with `depth` questions in a row, and the percent of the outcome's spread it explains."""
    fitted_tree = DecisionTreeRegressor(max_depth=depth).fit(table, outcome)
    left_over = np.sum((outcome - fitted_tree.predict(table)) ** 2)
    return fitted_tree, 100 * (1 - left_over / np.sum((outcome - outcome.mean()) ** 2))


figure, axes = plt.subplots(1, 2, figsize=(11, 2.7))
greedy_results = []
for axis, seed in zip(axes, [4, 2]):                 # two samples of the same rule
    table, outcome = agree_sample(seed)
    one_question, share_one = fit_share(table, outcome, 1)
    two_questions, share_two = fit_share(table, outcome, 2)
    greedy_results.append((share_one, share_two))
    grid_a, grid_b = np.meshgrid(np.linspace(-1, 1, 200), np.linspace(-1, 1, 200))
    shaded = two_questions.predict(pd.DataFrame({"A": grid_a.ravel(), "B": grid_b.ravel()}))
    axis.pcolormesh(grid_a, grid_b, shaded.reshape(grid_a.shape), cmap="RdBu", vmin=-1.5,
                    vmax=1.5, shading="auto", alpha=0.6)
    axis.scatter(table["A"], table["B"], c=outcome, cmap="RdBu", vmin=-1.5, vmax=1.5, s=14,
                 edgecolor="gray", linewidth=0.3)
    first_cut = one_question.tree_.threshold[0]
    if one_question.tree_.feature[0] == 0:
        axis.axvline(first_cut, color="black", linewidth=4)
    else:
        axis.axhline(first_cut, color="black", linewidth=4)
    axis.set_title(f"one question: {share_one:.0f}%   two: {share_two:.0f}%", fontsize=17)
    axis.set_xlabel("A")
    axis.set_ylabel("B")
plt.show()

- A rule that needs two characteristics at once: high when A and B
  agree, low when they disagree.
- **Greedy** means the first question is the best **on its own**. Here
  that is almost nothing, so the cut is nearly a coin flip: the black
  line.
- Left, the cut lands near the middle and two questions find the rule.
  Right, it lands at the edge and they do not.

## A World We Control

In [ ]:
# A simulated world of stocks, one row per stock-month. Six characteristics, each spread evenly
# between -1 and +1 (Gu, Kelly & Xiu rank theirs the same way): size, value, momentum and
# three columns of pure noise.
CHARACTERISTICS = ["size", "value", "momentum", "noise 1", "noise 2", "noise 3"]


def make_world(n_rows, generator, signal=0.06):
    """n_rows stock-months. Two rules: value pays only among small firms (size below 0), and
    momentum pays a little everywhere. Returns are simple, in decimals per month, with noise
    of 10% a month (sd)."""
    world = pd.DataFrame(generator.uniform(-1, 1, size=(n_rows, 6)), columns=CHARACTERISTICS)
    value_pays = np.where(world["size"] < 0, world["value"], 0.0)
    world["expected"] = signal * value_pays + signal / 3 * world["momentum"]
    world["return"] = world["expected"] + generator.normal(0, 0.10, size=n_rows)
    return world


generator = np.random.default_rng(2026)
fitting = make_world(6000, generator)       # the data the models learn from
new = make_world(20000, generator)          # new data, to score the forecasts


def new_data_r2(forecasts):
    """Out-of-sample R², %: squared errors on new data against the fitting data's average."""
    benchmark = fitting["return"].mean()
    errors = new["return"] - forecasts
    return 100 * (1 - np.sum(errors ** 2) / np.sum((new["return"] - benchmark) ** 2))


best_possible = new_data_r2(new["expected"])     # forecast with the truth itself

# The truth on a map of size and value (momentum at 0).
size_grid, value_grid = np.meshgrid(np.linspace(-1, 1, 201), np.linspace(-1, 1, 201))
truth_map = 100 * 0.06 * np.where(size_grid < 0, value_grid, 0)
plt.figure(figsize=(10, 2.4))
plt.pcolormesh(size_grid, value_grid, truth_map, cmap="RdBu", vmin=-6, vmax=6, shading="auto")
plt.colorbar(label="expected, %")
plt.text(-0.5, 0, "small firms:\nvalue pays", ha="center", va="center")
plt.text(0.5, 0, "big firms:\nvalue does not", ha="center", va="center")
plt.xlabel("size")
plt.ylabel("value")
plt.show()

- 6,000 stock-months to fit, 20,000 new ones to score. Noise: 10% a
  month.
- **Two rules:** value pays only among small firms; momentum pays a
  little everywhere. The other three columns are pure noise.
- We know the truth, so we know the best possible new-data $R^2$. Even
  knowing the true rule cannot predict the random noise.

## The Tree Finds the Rule

In [ ]:
# One tree of depth 2, grown on two characteristics only: size and value.
world_tree = DecisionTreeRegressor(max_depth=2)
world_tree.fit(fitting[["size", "value"]], fitting["return"])

figure, axes = plt.subplots(1, 2, figsize=(11, 3.7), gridspec_kw={"width_ratios": [1.5, 1]})
draw_tree(world_tree, axes[0])
shade_leaves(world_tree, axes[1], "size", (-1, 1), "value", (-1, 1),
            colors="RdBu", limits=(-4.5, 4.5))
plt.show()

- It asks about value first, then about size, and cuts both near 0: the
  rule we hid.
- The small firms’ leaves are far from zero; the big firms’ are near it.

## Deep Trees Memorize

In [ ]:
def fitted_r2(forecasts):
    """In-sample R², %: how well forecasts fit the fitting data itself."""
    actual = fitting["return"]
    return 100 * (1 - np.sum((actual - forecasts) ** 2) / np.sum((actual - actual.mean()) ** 2))


X_fit, y_fit, X_new = fitting[CHARACTERISTICS], fitting["return"], new[CHARACTERISTICS]
depths = list(range(1, 13))
fit_by_depth, new_by_depth = [], []
for depth in depths + [None]:                 # None: no limit, split to one stock per leaf
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_fit, y_fit)
    fit_by_depth.append(fitted_r2(tree.predict(X_fit)))
    new_by_depth.append(new_data_r2(tree.predict(X_new)))

plt.figure(figsize=(10, 2.7))
plt.plot(depths, fit_by_depth[:-1], color="tab:blue", marker="o", linewidth=3)
plt.plot(depths, new_by_depth[:-1], color="tab:orange", marker="o", linewidth=3)
plt.axhline(0, color="black", linewidth=0.8)
plt.axhline(best_possible, color="gray", linestyle="--")
plt.text(12.3, fit_by_depth[-2], "fit", color="tab:blue", va="center")
plt.text(12.3, new_by_depth[-2], "new data", color="tab:orange", va="center")
plt.text(0.8, best_possible + 2, "best possible", color="gray")
plt.xlim(0.5, 14.5)
plt.xlabel("maximum depth of the tree")
plt.ylabel("R² (%)")
plt.show()

- Deeper fits better and forecasts worse.
- Reading the best depth off the new data would be peeking: a validation
  block picks it.
- With no limit, the fit is perfect and the new-data $R^2$ is far below
  zero.

## The Smallest Leaf: A Second Dial

In [ ]:
# No depth limit, but every leaf must hold at least leaf_size stock-months (min_samples_leaf).
leaf_sizes = [20, 50, 100, 200, 300, 500, 1000, 2000]
fit_by_leaf, new_by_leaf, count_by_leaf = [], [], []
for leaf_size in leaf_sizes:
    tree = DecisionTreeRegressor(min_samples_leaf=leaf_size, random_state=0).fit(X_fit, y_fit)
    fit_by_leaf.append(fitted_r2(tree.predict(X_fit)))
    new_by_leaf.append(new_data_r2(tree.predict(X_new)))
    count_by_leaf.append(tree.get_n_leaves())      # how many leaves the tree ends up with

plt.figure(figsize=(10, 2.3))
plt.plot(leaf_sizes, fit_by_leaf, color="tab:blue", marker="o", linewidth=3)
plt.plot(leaf_sizes, new_by_leaf, color="tab:orange", marker="o", linewidth=3)
plt.axhline(0, color="black", linewidth=0.8)
plt.axhline(best_possible, color="gray", linestyle="--")
plt.xscale("log")                                  # each step on the axis multiplies the size
labeled = [size for size in leaf_sizes if size != 300]   # a 300 label would crowd the 200
plt.xticks(labeled, [f"{size:,}" for size in labeled])
plt.minorticks_off()
plt.text(17, fit_by_leaf[0], "fit", color="tab:blue", ha="right", va="center")
plt.text(17, new_by_leaf[0], "new data", color="tab:orange", ha="right", va="center")
plt.text(2200, best_possible + 2, "best possible", color="gray", ha="right")
plt.xlim(6, 2600)
plt.xlabel("fewest stock-months a leaf may hold")
plt.ylabel("R² (%)")
plt.show()

- No depth limit, but every leaf must hold at least so many stock-months
  (`min_samples_leaf`), so each leaf’s average rests on enough rows.
- Small leaves memorize; big leaves are too coarse to find the rule.
- The best size is in between. Reading it off the new data would be
  peeking: a validation block picks it, as it picks the depth.

## Depth Is the Dial

- **Maximum depth** (`max_depth`) and the **fewest rows a leaf may
  hold** (`min_samples_leaf`) are tuning parameters.
- Choose them on validation data, as in the regularization deck’s “Every
  Method Has a Dial”.
- One tree is easy to read, but coarse: every stock in a leaf gets the
  same forecast, and a small change in the data can change the
  questions.
- Many small trees fix that. Read: ISLP 8.1; ESL 9.2.

# Trees for Yes or No

## Number or Class?

- **Regression:** the target is a number, such as next month’s return. A
  tree leaf forecasts the **average** of its rows.
- **Classification:** the target is a class, such as default or repay,
  fraud or not. A tree leaf forecasts the **share** of each class: “20%
  of loans like this default.”
- A share is a probability. A **cutoff** turns it into a decision:
  refuse the loan above 25%?
- The same trees, forests and boosting handle both. Boosted trees are a
  standard tool in credit scoring and fraud detection.
- Next: a classification problem, with real data.

## Will This Loan Default?

- **One row** is one finished loan from Lending Club, issued 2012–2013.
  **Default** means charged off: about one loan in six.
- **Inputs** known when the loan was made: FICO credit score, term (36
  or 60 months), debt-to-income, loan size, income.
- A 60-month loan defaults more than twice as often as a 36-month loan.
- The code for this slide is in the slides only: the loan file is not
  part of this notebook.

## Logit: A Curve Between 0 and 1

- A straight line can forecast below 0 or above 1. We want a
  **probability**.
- **Logistic function:** $p = 1/(1 + e^{-z})$ turns any number $z$ into
  a value between 0 and 1, an S-shaped curve.
- **Logit:** a regression for $z = a + b \cdot \text{FICO}$, then the
  logistic function.

## A Tree for Default

- Start at the top, answer each yes-or-no question, and use the default
  rate in the final leaf.
- The first question is about the term: the cut at 48 separates 36-month
  loans from 60-month loans. Then FICO on both sides, at nearly the same
  cut.
- **The interaction is in the leaves:** a good score lowers default more
  on 60-month loans than on 36-month loans.
- No leaf passes 50%, so “default or not” needs a cutoff below that.

## The ROC Curve

- **Score:** the default rate of the loan’s leaf, for loans held out
  from fitting.
- Predict “default” when the score exceeds a threshold. **True positive
  rate:** share of defaults predicted. **False positive rate:** share of
  repaid loans predicted to default.
- **ROC curve:** the two rates as the threshold falls, leaf by leaf.
  Diagonal: a random classifier.
- **AUC**, the area under the curve: the probability that a random
  default scores above a random repaid loan (ties count half).

## A Grown Tree Memorizes Loans

- **No depth limit:** the tree keeps asking until no leaf can be split.
- On its own loans it ranks almost perfectly: it has memorized who
  defaulted.
- On new loans it is barely above a coin flip, and below the depth-2
  tree.
- As on “Deep Trees Memorize”, choose the depth on validation data.

# Many Trees

## One Tree Wobbles

In [ ]:
# One characteristic x and a curved truth, with few rows and much noise: 80 observations, noise
# sd 0.6. That is where a single tree is least stable (the boosting slides use 300 rows).
wobble_generator = np.random.default_rng(5)
x_few = np.sort(wobble_generator.uniform(-1, 1, 80))
y_few = np.sin(3 * x_few) + wobble_generator.normal(0, 0.6, 80)
X_few = x_few.reshape(-1, 1)                      # scikit-learn wants a table: one column
x_grid = np.linspace(-1, 1, 400)                  # where the forecasts are drawn and scored
X_grid = x_grid.reshape(-1, 1)
truth_grid = np.sin(3 * x_grid)


def resample_rows(number):
    """A resample: 80 draws from the 80 rows, with replacement (some rows twice, some never)."""
    return np.random.default_rng(number).integers(0, 80, 80)


figure, axes = plt.subplots(1, 4, figsize=(11, 2.4), sharey=True)
for number, axis in enumerate(axes):
    rows = resample_rows(number)
    tree = DecisionTreeRegressor(max_depth=4).fit(X_few[rows], y_few[rows])
    axis.scatter(x_few[rows], y_few[rows], color="lightgray", s=12)
    axis.plot(x_grid, truth_grid, color="black", linestyle="--", linewidth=2)
    axis.plot(x_grid, tree.predict(X_grid), color="tab:orange", linewidth=3)
    axis.set_title(f"resample {number + 1}", fontsize=16)
    axis.set_xticks([])
    axis.set_yticks([])
plt.subplots_adjust(wspace=0.05)
plt.show()

- Four **resamples** of the same 80 noisy rows, each with its own tree
  (4 questions deep).
- **With replacement:** a resample of rows 1 to 6 might be 2, 2, 3, 5,
  6, 6. Rows 2 and 6 come twice; rows 1 and 4 not at all.
- Dashed: the truth. Orange: four very different staircases, each
  chasing its own noise. A small change in the data changes the
  questions: one tree is **unstable**.

## Average Many Trees

In [ ]:
# 200 trees, each fitted to its own resample, and their average ("bagging": bootstrap aggregation).
bag_forecasts = np.array([
    DecisionTreeRegressor(max_depth=4).fit(X_few[resample_rows(number)],
                                           y_few[resample_rows(number)]).predict(X_grid)
    for number in range(200)])
bagged = bag_forecasts.mean(axis=0)
one_tree = DecisionTreeRegressor(max_depth=4).fit(X_few, y_few).predict(X_grid)


def miss_from_truth(forecasts):
    """Typical miss against the true curve (root mean square), in the units of y."""
    return np.sqrt(np.mean((forecasts - truth_grid) ** 2))


figure, axes = plt.subplots(1, 2, figsize=(11, 2.4), sharey=True)
for number in range(5):
    axes[0].plot(x_grid, bag_forecasts[number], linewidth=1.5, alpha=0.8)
axes[0].plot(x_grid, truth_grid, color="black", linestyle="--", linewidth=2)
axes[0].set_title("5 of the 200 trees", fontsize=16)
axes[1].plot(x_grid, one_tree, color="gray", linewidth=2, label="one tree")
axes[1].plot(x_grid, bagged, color="tab:orange", linewidth=3, label="average of 200")
axes[1].plot(x_grid, truth_grid, color="black", linestyle="--", linewidth=2)
axes[1].legend(frameon=False, fontsize=14, loc="lower right")
axes[1].set_title("the average", fontsize=16)
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
plt.subplots_adjust(wspace=0.05)
plt.show()

- Fit a tree to each of 200 resamples, then **average** the 200
  forecasts: **bagging**.
- Each tree is noisy in its own way, so the noise cancels: the average
  is closer to the truth than one tree.
- Week 2’s “average the forecasts”, with trees.

## Trees That Differ: Forests

In [ ]:
# In the simulated world, which characteristic does each of 100 resampled trees ask about first?
def first_questions(max_features):
    """How often each characteristic is a tree's first question, over 100 resampled trees.
    max_features: how many characteristics a question may look at (None: all six)."""
    counts = pd.Series(0, index=CHARACTERISTICS)
    for number in range(100):
        rows = np.random.default_rng(number).integers(0, len(X_fit), len(X_fit))
        tree = DecisionTreeRegressor(max_depth=3, max_features=max_features, random_state=number)
        tree.fit(X_fit.iloc[rows], y_fit.iloc[rows])
        counts[CHARACTERISTICS[tree.tree_.feature[0]]] += 1
    return counts


all_available = first_questions(None)
two_at_random = first_questions(2)

figure, axes = plt.subplots(1, 2, figsize=(11, 3.1), sharex=True)
for axis, counts, title in [(axes[0], all_available, "every question sees all 6"),
                            (axes[1], two_at_random, "every question sees 2 at random")]:
    axis.barh(CHARACTERISTICS, counts.to_numpy(), color="tab:blue")
    axis.set_title(title, fontsize=16)
    axis.tick_params(axis="y", labelsize=15)
    axis.invert_yaxis()
    axis.set_xlabel("trees (of 100) asking it first")
    for side in ["top", "right"]:
        axis.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

- Resampling alone is not enough: if every question may use every
  characteristic, nearly all trees ask about the same one first. Near
  copies make the same mistakes, so averaging cancels fewer of them.
- **Random forest:** each question sees only a random few of the
  characteristics. Each tree is a little worse alone, but the trees now
  differ, so more of their mistakes cancel.
- Two ways to make trees differ: **different rows and characteristics**
  (forest), or **a different target** (boosting).

## How a Random Forest Is Grown

- **Resample:** each tree gets its own resample of the rows, as in
  bagging.
- **Offer a few:** at every question, the tree may use only a few
  characteristics, drawn at random (here 2 of the 6), and asks the best
  question among them.
- **Average:** grow hundreds of deep trees this way and average their
  forecasts. For a yes-or-no target, average the leaves’ shares.
- **Dials:** how many characteristics each question is offered; the
  smallest leaf. Adding trees does not overfit; it only takes longer.

# Boosting

## One Round of Boosting

In [ ]:
# One characteristic x, a curved truth, and 300 noisy observations (the same curve as "One Tree
# Wobbles", with more rows and less noise).
curve_generator = np.random.default_rng(5)
x_curve = np.sort(curve_generator.uniform(-1, 1, 300))
truth_curve = np.sin(3 * x_curve)
y_curve = truth_curve + curve_generator.normal(0, 0.3, 300)
X_curve = x_curve.reshape(-1, 1)                  # scikit-learn wants a table: one column

# One round, with a full step so that it can be seen. Start from the average; the leftovers
# (residuals) are what that forecast still gets wrong; a small tree is fitted to the leftovers.
start = np.full(300, y_curve.mean())
leftovers = y_curve - start
small_tree = DecisionTreeRegressor(max_depth=2).fit(X_curve, leftovers)
after = start + small_tree.predict(X_curve)

figure, axes = plt.subplots(1, 3, figsize=(11, 2.5), sharey=True)
axes[0].scatter(x_curve, y_curve, color="lightgray", s=10)
axes[0].vlines(x_curve, start, y_curve, color="tab:red", alpha=0.35, linewidth=1)
axes[0].plot(x_curve, start, color="tab:orange", linewidth=3)
axes[0].set_title("1. start: the average", fontsize=15)
axes[1].scatter(x_curve, leftovers, color="tab:red", alpha=0.5, s=10)
axes[1].plot(x_curve, small_tree.predict(X_curve), color="black", linewidth=3)
axes[1].set_title("2. fit the leftovers", fontsize=15)
axes[2].scatter(x_curve, y_curve, color="lightgray", s=10)
axes[2].vlines(x_curve, after, y_curve, color="tab:red", alpha=0.35, linewidth=1)
axes[2].plot(x_curve, after, color="tab:orange", linewidth=3)
axes[2].set_title("3. add the tree", fontsize=15)
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
plt.subplots_adjust(wspace=0.05)
plt.show()
leftover_before = np.mean(leftovers ** 2)
leftover_after = np.mean((y_curve - after) ** 2)

- Gray: the data. Red lines: the **leftovers**, what the forecast still
  gets wrong. A small tree is fitted to the leftovers, not to the data,
  and added to the forecast.
- The leftovers shrink. **Repeat**, adding only a small fraction of each
  tree (0.1: the **learning rate**).
- **Why a different tree every time:** each one is fitted to what the
  trees before it left over.

## Boosting, Tree by Tree

In [ ]:
# The curve data (x_curve, y_curve, truth_curve) are from "One Round of Boosting".
forecast = np.full(300, y_curve.mean())           # 1. start: every forecast is the average
snapshots = {}
for tree_number in range(1, 101):
    residual = y_curve - forecast                 # 2. what the forecast still gets wrong
    stump = DecisionTreeRegressor(max_depth=1).fit(X_curve, residual)   # 3. a small tree
    forecast = forecast + 0.1 * stump.predict(X_curve)                  # 4. a small step
    if tree_number in (1, 10, 100):
        snapshots[tree_number] = forecast.copy()

figure, axes = plt.subplots(1, 3, figsize=(11, 2.8), sharey=True)
for axis, count in zip(axes, [1, 10, 100]):
    axis.scatter(x_curve, y_curve, color="lightgray", s=10)
    axis.plot(x_curve, truth_curve, color="black", linestyle="--", linewidth=2)
    axis.plot(x_curve, snapshots[count], color="tab:orange", linewidth=3)
    axis.set_title(f"after {count} tree" + ("s" if count > 1 else ""))
    axis.set_xticks([])
    axis.set_yticks([])
axes[0].set_ylabel("y")
plt.subplots_adjust(wspace=0.05)
plt.show()

- Orange: the forecast, a staircase. Dashed: the truth. After 1 tree, a
  small step away from the average; after 100, close to the curve.

## Your Turn: Bigger Steps

- The same loop, 300 trees. Try a learning rate of 0.1, then 1.0. Which
  stays closer to the truth?

``` python
step = ____                              # the learning rate
forecast = np.full(300, y_curve.mean())
for tree_number in range(300):
    residual = y_curve - forecast
    stump = DecisionTreeRegressor(max_depth=1)
    stump.fit(X_curve, residual)
    forecast = forecast + step * stump.predict(X_curve)
miss = np.mean((forecast - truth_curve) ** 2)
print(f"step {step}: squared miss {miss:.4f}")
```

## Three Dials

- **Depth of each tree** (`max_depth`): 1 asks one question; 2 lets a
  second question depend on the first. Boosted one-question trees add up
  one characteristic at a time, as straight lines do: they cannot find
  “value pays among small firms”.
- **Learning rate** (`learning_rate`): the size of each step. Smaller
  steps need more trees; shrinkage again.
- **Number of trees** (`n_estimators`): how many steps. Too many, and
  boosting fits the noise; more trees never make a forest overfit.
- All three are arguments of scikit-learn’s `GradientBoostingRegressor`.
  Its defaults are depth 3, learning rate 0.1 and 100 trees, not tuned
  for returns. Read: ESL 10.11–10.12.

# Boosted Trees at Work

## Review: What a Tree Forecasts

- A tree asks yes-or-no questions about a stock’s characteristics.
- The answers send the stock to one **leaf**.
- Its forecast is the **average** return of the stocks that fell in that
  leaf when the tree was grown.
- Each question is the one that leaves the **least** squared gap inside
  its two sides.

## Review: Why Deep Trees Disappoint

- Every extra question splits the leaves again. With no depth limit,
  each stock ends in a leaf of its own.
- That tree fits the data it was grown on perfectly, noise included: it
  **memorizes**.
- On new data it forecasts worse than a shallower tree.
- Two dials hold it back, the **depth** and the **smallest leaf**,
  chosen on validation data, never on the data the tree was grown on.

## Review: Why Averaging Trees Helps

- One tree is noisy: grown on other rows, it asks other questions.
- Grow many trees, each on its own **resample** of the rows, and
  **average** their forecasts: the noise mostly cancels.
- A **forest** also lets each question use only a few characteristics,
  picked at random, so the trees differ more.
- More trees steady the average. They do not make a forest overfit.

## Review: Forest or Boosting?

- **Forest:** deep trees grown side by side, each on its own resample;
  the forecast is their average.
- **Boosting:** small trees grown one after another, each fitted to what
  the trees so far still get wrong; each adds a small step.
- Too many boosted trees fit the noise. More trees never hurt a forest.

## Upload, Then Run Setup

Starting a new session here? Upload the four files
(`size_value_monthly.csv`, `gw_monthly.csv`, `ames_housing.csv`,
`polish_bankruptcy.csv.gz`), click the cell where you pick up, then
*Runtime → Run before*: it runs Setup, the helpers and every cell above
it, which the rest needs (the simulated world, the curve, the fitted
models).

# Trees, Step by Step

## Growing a Tree, Step by Step

1.  Put all the rows in one leaf.
2.  Try every characteristic and every cut: split the leaf in two, and
    forecast each side by its own average.
3.  Add up the squared gaps left inside the two sides. Keep the question
    that leaves the **least**.
4.  Repeat steps 2 and 3 inside each new leaf.
5.  Stop at the depth limit, or when a leaf would get too few rows. A
    new row’s forecast: the average of the leaf it lands in.

## Predict: Which Question First?

In [ ]:
# Eight made-up stocks, few enough to work by hand. size: 1 = small, 2 = large. value: 1 =
# growth (expensive) to 4 = value (cheap). return: next month's return in PERCENT, whole
# numbers, so the arithmetic can be done with a calculator.
eight = pd.DataFrame({"size": [1, 1, 1, 1, 2, 2, 2, 2],
                      "value": [1, 2, 3, 4, 1, 2, 3, 4],
                      "return": [1, 3, 7, 9, 2, 2, 3, 5]}, index=list("ABCDEFGH"))
print(eight.T.to_string())

- Eight made-up stocks. **Size:** 1 small, 2 large. **Value:** 1 growth
  (expensive) to 4 value (cheap). **Return:** next month, in percent.
- **Predict first:** which question does a tree ask first?
  1.  Size ≤ 1?
  2.  Value ≤ 1?
  3.  Value ≤ 2?
  4.  Value ≤ 3?

## The First Question, by Hand

In [ ]:
def eight_left_over(column, cut):
    """The squared gaps of the returns left inside the two sides of 'column <= cut?',
    added up. Each side is forecast by its own average."""
    yes = eight[eight[column] <= cut]["return"]
    no = eight[eight[column] > cut]["return"]
    return np.sum((yes - yes.mean()) ** 2) + np.sum((no - no.mean()) ** 2)


eight_spread = np.sum((eight["return"] - eight["return"].mean()) ** 2)   # before any question
for column, cut in [("size", 1), ("value", 1), ("value", 2), ("value", 3)]:
    left = eight_left_over(column, cut)
    question = f"{column} ≤ {cut}?"
    print(f"{question:<11} left over {left:4.1f}   "
          f"explains {100 * (1 - left / eight_spread):3.0f}%")

- Before any question: the spread is the eight returns’ squared gaps
  from their average, added up.
- For each question, forecast each side by its own average and add up
  the squared gaps left inside the two sides. The share of the spread
  that is gone is the share explained.
- The tree asks the question that leaves the least.

## The Second Question

In [ ]:
# The same steps inside each side of the first question. scikit-learn tries every question.
eight_tree = DecisionTreeRegressor(max_depth=2).fit(eight[["size", "value"]], eight["return"])
figure, axis = plt.subplots(figsize=(10, 2.3))
draw_tree(eight_tree, axis, leaf_text=lambda number, node:
          f"leaf {number}\n{eight_tree.tree_.value[node][0][0]:g}%")
plt.show()

- **Value side** (C, D, G, H): the size question leaves the least, so
  the tree asks about size.
- **Growth side** (A, B, E, F): the tree asks “value ≤ 1?”.
- Size matters among value stocks and not among growth stocks: an
  **interaction**.

## Boosting, Step by Step

1.  **Start:** every row’s forecast is the average.
2.  **Leftovers:** each row’s actual value minus its forecast.
3.  Grow a **small tree on the leftovers**, not on the actual values:
    the steps of one tree, with a small depth.
4.  **Step:** add the learning rate × that tree’s forecast to every
    forecast.
5.  **Repeat** steps 2 to 4, once for each tree.

## Exercise: Trees by Hand

- **In pairs, 15 minutes.** The three tasks are the next three sections
  of the notebook. Go in order, as far as you get.

1.  **Does the Tree Work Next Month?** Score the tree of “The Second
    Question” on next month’s returns: forecasts, squared gaps, total.
2.  **One Round of Boosting:** start at the average, ask “value ≤ 2?”,
    take half a step. Fill in the leftovers, the small tree, the
    forecasts.
3.  **The Next Tree’s Question:** run `next_round` for all four
    questions. Which leaves the least? Is it round 1’s question again?

- **Check yourself:** after you fill in a cell, run the cell below it.
  It names the stocks to look at again.

## Exercise: Does the Tree Work Next Month?

- The eight stocks again, with next month’s returns (made up). Does the
  tree grown on the first month still forecast well?

1.  Scroll up: the tree is drawn on “The Second Question”; each stock’s
    size and value are in the table of “Predict: Which Question First?”.
2.  Send each stock down the tree, answering its questions with the
    stock’s size and value. The number in its leaf is its forecast. Type
    the eight forecasts, A to H, in `my_tree_says`.
3.  For each stock: next month’s return minus its forecast, squared.
    Type the eight in `my_gaps`.
4.  Add up the eight squared gaps: that is `my_miss`.
5.  Run the cell, then the check cell below it. If it names a stock,
    look at that stock again.

In [ ]:
# Next month's returns of the stocks A to H, in percent.
next_month = [3, 1, 9, 8, 1, 3, 5, 3]
# Replace each ... with a number.
# The tree of "The Second Question": each stock's forecast.
my_tree_says = [..., ..., ..., ..., ..., ..., ..., ...]
# The squared gap of each forecast from next month's return.
my_gaps = [..., ..., ..., ..., ..., ..., ..., ...]
# The gaps, added up.
my_miss = ...

In [ ]:
# Checks your numbers in the cell above: run it after you fill them in. It names what to look
# at again and never prints the right numbers.
tree_says = eight_tree.predict(eight[["size", "value"]])      # the tree of the last slide
right_gaps = (np.array(next_month) - tree_says) ** 2


def check_row(name, yours, right):
    """Print 'right', or what to look at again. It never prints the right numbers."""
    yours = yours if isinstance(yours, list) else [yours]       # one number or eight
    right = list(np.atleast_1d(right))
    if ... in yours:
        print(f"{name}: replace each ... with a number")
    elif len(yours) != len(right):
        print(f"{name}: {len(right)} numbers, please, one per stock")
    else:
        look_again = [stock for stock, mine, true in zip(eight.index, yours, right)
                      if abs(mine - true) > 0.001]
        if not look_again:
            print(f"{name}: right")
        elif len(right) == 1:
            print(f"{name}: not yet")
        else:
            stocks = "stock" if len(look_again) == 1 else "stocks"
            print(f"{name}: look again at {stocks} {', '.join(look_again)}")


check_row("the tree's forecasts", my_tree_says, tree_says)
check_row("squared gaps", my_gaps, right_gaps)
check_row("the miss", my_miss, right_gaps.sum())

## Exercise: One Round of Boosting

- The eight stocks again: one round of “Boosting, Step by Step”, by
  hand, with a learning rate of 0.5. Their returns are in the table of
  “Predict: Which Question First?”.

1.  **Start:** every stock’s forecast is the average return, 4.
2.  **Leftovers:** each stock’s return minus 4. Type the eight, A to H,
    in `my_leftovers`.
3.  **The small tree** asks “value ≤ 2?”. Its forecast on each side is
    the average leftover there: A, B, E, F (yes) in `my_tree_yes`; C, D,
    G, H (no) in `my_tree_no`.
4.  **New forecasts:** 4 + 0.5 × the small tree’s forecast for the
    stock’s side. Type the eight, A to H, in `my_forecasts`.
5.  Run the cell, then the check cell below it. If it names a stock,
    look at that stock again.

In [ ]:
# Round 1 by hand. Replace each ... with a number.
# The stocks in order: A B C D E F G H.
my_leftovers = [..., ..., ..., ..., ..., ..., ..., ...]
my_tree_yes = ...    # average leftover of A, B, E, F
my_tree_no = ...     # average leftover of C, D, G, H
my_forecasts = [..., ..., ..., ..., ..., ..., ..., ...]

In [ ]:
# Checks your numbers in the cell above: run it after you fill them in. The right numbers
# follow the steps of "Boosting, Step by Step"; the check names what to look at again.
right_start = eight["return"].mean()                          # 1. start: the average
right_leftovers = eight["return"] - right_start               # 2. leftovers
round1_yes = eight["value"] <= 2                              # 3. the small tree's question
right_tree_yes = right_leftovers[round1_yes].mean()           #    its forecast on each side
right_tree_no = right_leftovers[~round1_yes].mean()
right_forecasts = right_start + 0.5 * np.where(round1_yes, right_tree_yes, right_tree_no)  # 4.


# check_row, which prints "right" or what to look at again, is defined in the check cell of
# "Exercise: Does the Tree Work Next Month?".

check_row("leftovers", my_leftovers, right_leftovers)
check_row("small tree, yes side", my_tree_yes, right_tree_yes)
check_row("small tree, no side", my_tree_no, right_tree_no)
check_row("new forecasts", my_forecasts, right_forecasts)
if ... not in my_forecasts and len(my_forecasts) == 8:      # how far off YOUR forecasts are
    my_off = np.sum((eight["return"] - np.array(my_forecasts)) ** 2)
    print(f"Your forecasts are off by {my_off:g} in total "
          f"(the average alone: {np.sum(right_leftovers ** 2):g}).")

## Exercise: The Next Tree’s Question

- Round 2 of boosting: the next small tree is grown on what round 1 left
  over. Which question does it ask?

1.  Run the cell below. It sets up `next_round(column, cut)`: round 1,
    then round 2 with the question “column ≤ cut?”. It prints how far
    off the new forecasts are in total. Its example: “value ≤ 1?”.
2.  Click **+ Code** and run the other three, one per line:
    `next_round("size", 1)`, `next_round("value", 2)` and
    `next_round("value", 3)`.
3.  Which question leaves the least off in total? The next tree asks it.
4.  Is it round 1’s question, “value ≤ 2?”, again?

In [ ]:
# Round 2 for any question: round 1 first, then a one-question tree fitted to round 1's new
# leftovers, then half a step. It prints how far off the new forecasts are: the squared
# leftovers, added up.
def next_round(column, cut):
    """Round 2 of boosting with the question 'column <= cut?' on the eight stocks."""
    # Round 1, as in "Boosting, Step by Step": start from the average, fit a one-question
    # tree to the leftovers, take half a step.
    columns = eight[["size", "value"]]
    start = eight["return"].mean()
    first_tree = DecisionTreeRegressor(max_depth=1).fit(columns, eight["return"] - start)
    after_round_1 = start + 0.5 * first_tree.predict(columns)
    # Round 2: the question 'column <= cut?' on what round 1 left over.
    leftovers = eight["return"] - after_round_1
    yes = eight[column] <= cut
    tree_says = np.where(yes, leftovers[yes].mean(), leftovers[~yes].mean())
    new_forecasts = after_round_1 + 0.5 * tree_says             # half a step
    off_in_total = np.sum((eight["return"] - new_forecasts) ** 2)
    print(f"{column} ≤ {cut}?  off in total {off_in_total:g}")


next_round("value", 1)

## Next Month: The Deepest Tree Slips

In [ ]:
# Four trees grown on the first month's returns, each scored on both months: the squared gaps,
# added up. Questions asked: 0 is the average alone; 3 gives every stock a leaf of its own.
eight_columns_now = eight[["size", "value"]]
misses = {}
for questions in [0, 1, 2, 3]:
    if questions == 0:
        forecasts = np.full(8, eight["return"].mean())
    else:
        grown = DecisionTreeRegressor(max_depth=questions).fit(eight_columns_now, eight["return"])
        forecasts = grown.predict(eight_columns_now)
    misses[questions] = (np.sum((eight["return"] - forecasts) ** 2),
                         np.sum((np.array(next_month) - forecasts) ** 2))
miss_table = pd.DataFrame(misses, index=["first month", "next month"])
miss_table.columns.name = "questions asked"
print(miss_table.to_string(float_format=lambda number: f"{number:g}"))

- Next month the two-question tree misses least.
- The tree with a leaf for every stock fits the first month perfectly,
  and next month it misses almost three times as much as the
  two-question tree.
- It fits the first month perfectly and does worse next month than the
  two-question tree: a better fit to old data is not a better forecast.
- So a tree stops: at a depth limit, or when a leaf would hold too few
  stocks.

## One Round of Boosting, by Hand

In [ ]:
# One round on the eight stocks, with a learning rate of 0.5: half a step, so that the step
# shows and the arithmetic stays in whole numbers.
eight_columns = eight[["size", "value"]]
eight_start = np.full(8, eight["return"].mean())               # 1. start: the average
eight_leftover = eight["return"] - eight_start                 # 2. what it still gets wrong
eight_stump = DecisionTreeRegressor(max_depth=1).fit(eight_columns, eight_leftover)  # 3.
eight_after = eight_start + 0.5 * eight_stump.predict(eight_columns)   # 4. half a step
one_round = pd.DataFrame({"return": eight["return"], "forecast": eight_start,
                          "leftover": eight_leftover,
                          "small tree": eight_stump.predict(eight_columns),
                          "new forecast": eight_after,
                          "new leftover": eight["return"] - eight_after})
print(one_round.T.to_string(float_format=lambda number: f"{number:g}"))

- A one-question tree fitted to the **leftovers** asks “value ≤ 2?”.
- Learning rate 0.5: each forecast moves half of that tree’s forecast.
  The squared leftovers shrink.

## The Next Tree, Checked

In [ ]:
# All four questions for the next tree. Then scikit-learn's two rounds, as a check: it tries
# every question in each round by itself.
for column, cut in [("size", 1), ("value", 1), ("value", 2), ("value", 3)]:
    next_round(column, cut)
two_rounds = GradientBoostingRegressor(max_depth=1, learning_rate=0.5, n_estimators=2)
two_rounds.fit(eight_columns, eight["return"])
two_rounds_off = np.sum((eight["return"] - two_rounds.predict(eight_columns)) ** 2)
print(f"scikit-learn, two rounds: off in total {two_rounds_off:g}")

- The next tree asks “value ≤ 3?”: stock D’s leftover is now the biggest
  miss.
- Round 1’s question again still helps, but less.
- Each tree fits what the trees before it left over, so the trees
  differ. scikit-learn’s two rounds agree.

## A Random Forest, Step by Step

1.  **Resample:** draw as many rows as the data has, at random, with
    replacement.
2.  **Grow a deep tree** on the resample. At each question, offer it
    only a few characteristics, drawn at random.
3.  **Repeat**, each tree on its own resample.
4.  **Forecast:** the average of all the trees’ forecasts (for yes or
    no: the average share).

- Each tree is wrong in its own way; the average cancels much of it.

## Check: What the Next Tree Fits

- Boosting started from the average and has grown its first small tree.
- **Fingers up:** what is the second tree fitted to?
  1.  The returns, on a fresh resample of the rows
  2.  What the forecasts so far still get wrong
  3.  The returns, with a few characteristics per question
  4.  The first tree’s forecasts, to smooth them

# A Race on Real Data

## Predict: Who Prices Homes Best?

- **One row** is one home sale in Ames, Iowa, 2006 to 2010: 1,460 sales,
  79 characteristics each (size, quality, age, neighborhood).
- **Target:** the sale price, in logs, so an error is a percent.
- **Score:** $R^2$ on homes each model did not see (5-fold
  cross-validation). Racing: a line, ridge, one tree, a forest,
  boosting.
- **Predict first:** which finishes first?
  1.  Linear regression on all 79
  2.  Ridge, the week 3 shrinker
  3.  One tree
  4.  Boosted trees

## The Race on Real Data

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Ames, Iowa home sales, 2006-2010 (Dean De Cock, Journal of Statistics Education, 2011).
# A blank in a feature column (no alley, no pool) means "none", so categories get "none".
homes = pd.read_csv(resolve_data_path("ames_housing.csv"))
log_price = np.log(homes.pop("SalePrice"))              # log dollars: an error is about a percent
word_columns = homes.select_dtypes(exclude="number").columns.tolist()
number_columns = [name for name in homes.columns if name not in word_columns]

dummies = make_pipeline(SimpleImputer(strategy="constant", fill_value="none"),
                        OneHotEncoder(handle_unknown="ignore"))
for_lines = ColumnTransformer([
    ("numbers", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), number_columns),
    ("words", dummies, word_columns)])
for_trees = ColumnTransformer([
    ("numbers", SimpleImputer(strategy="median"), number_columns), ("words", dummies, word_columns)])

racers = {
    "linear regression": make_pipeline(for_lines, LinearRegression()),
    "ridge": make_pipeline(for_lines, RidgeCV(alphas=np.logspace(-1, 3, 30))),
    "one tree, depth 4": make_pipeline(for_trees, DecisionTreeRegressor(
        max_depth=4, min_samples_leaf=10, random_state=0)),
    "random forest": make_pipeline(for_trees, RandomForestRegressor(
        n_estimators=200, min_samples_leaf=2, max_features=0.33, n_jobs=-1, random_state=0)),
    "boosted trees": make_pipeline(for_trees, GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.03, max_depth=3, subsample=0.8, random_state=0)),
}
folds = KFold(5, shuffle=True, random_state=0)


def homes_r2(forecasts):
    """R², %, of out-of-fold forecasts of log price: the forecast for each home comes from a
    model that never saw that home."""
    return 100 * (1 - np.sum((log_price - forecasts) ** 2)
                  / np.sum((log_price - log_price.mean()) ** 2))


homes_forecasts = {}                         # each model's out-of-fold forecast of log price
for name, racer in racers.items():          # about a minute on Colab
    homes_forecasts[name] = cross_val_predict(racer, homes, log_price, cv=folds)
homes_score = {name: homes_r2(forecast) for name, forecast in homes_forecasts.items()}

colors = ["gray", "gray", "tab:blue", "tab:orange", "tab:orange"]   # lines, one tree, ensembles
plt.figure(figsize=(10, 2.8))
plt.barh(list(homes_score), list(homes_score.values()), color=colors)
for position, score in enumerate(homes_score.values()):
    plt.text(score + 0.5, position, f"{score:.1f}%", va="center")
plt.gca().invert_yaxis()
plt.xlim(60, 100)
plt.xlabel("R² on homes the model did not see (%)")
plt.show()

- Boosting finishes first, the forest second. Ridge beats plain
  regression.
- One tree is last: coarse leaves lose to a line. Many trees beat the
  line.
- Typical home: the line and boosting miss by about the same percent.
  The $R^2$ gap is a few homes.

## Exercise: The Homes Up Close

- **In pairs, 15 minutes.** Go in order, as far as you get. Tasks 2 to 4
  need the race: run its cell first if you have not.

1.  **The First Question:** run the helper cell, then the pasted code
    once per column. Which question does a tree ask first?
2.  **Where the Models Go Wrong:** `biggest_misses` for the line, then
    for boosting. The worst single miss? What do the big misses share?
3.  **Set Aside the Odd Sales:** score the ordinary sales only. Does the
    gap between the line and boosting change? The order?
4.  **A Deeper Tree?** Depths 2, 4, 8 and no limit. Does any catch the
    line? Compare new homes with the homes it was fit to.

## Exercise: The Data and a Helper

- **Data:** home sales in Ames, Iowa, 2006 to 2010. **Target:** the sale
  price, in logs, so a gap is about a percent.
- **Three characteristics:** `OverallQual` (overall quality, 1 to 10),
  `GrLivArea` (living area, square feet) and `YearBuilt`.

1.  Run the cell below. It loads the homes and sets up
    `homes_left_over(column, cut)`: step 3 of “Growing a Tree, Step by
    Step” for the question “column ≤ cut?”, the squared gaps left inside
    its two sides. Its example: “YearBuilt ≤ 1950?”.
2.  Then go on to “Exercise: The First Question”.

In [ ]:
# Ames, Iowa home sales, 2006-2010 (Dean De Cock, Journal of Statistics Education, 2011).
home_sales = pd.read_csv(resolve_data_path("ames_housing.csv"))
home_log_price = np.log(home_sales["SalePrice"])       # log dollars: a gap is about a percent
# OverallQual: overall quality, 1 to 10. GrLivArea: living area, square feet. YearBuilt.
home_columns = ["OverallQual", "GrLivArea", "YearBuilt"]
home_spread = np.sum((home_log_price - home_log_price.mean()) ** 2)   # before any question


def homes_left_over(column, cut):
    """The squared gaps of log price left inside the two sides of 'column <= cut?', added up.
    Each side is forecast by its own average: step 3 of 'Growing a Tree, Step by Step'."""
    yes = home_log_price[home_sales[column] <= cut]
    no = home_log_price[home_sales[column] > cut]
    return np.sum((yes - yes.mean()) ** 2) + np.sum((no - no.mean()) ** 2)


print(f"{len(home_sales):,} homes. Spread of log price: {home_spread:.1f}")
print(f"For example, 'YearBuilt ≤ 1950?' leaves {homes_left_over('YearBuilt', 1950):.1f}")

## Exercise: The First Question

- Which question does a tree ask first about these homes? Two blocks of
  code, in the notebook below these steps.

1.  Click **+ Code** and paste the first block.
2.  Fill the blank `____`: the best cut leaves the least, so `np.argmin`
    needs the list of what each cut leaves over.
3.  Run it. It tries every cut of `OverallQual` and prints the best one,
    with the share of the spread it explains.
4.  Change `"OverallQual"` to `"GrLivArea"` and run it again; then to
    `"YearBuilt"`. Which question explains the most? A tree asks it
    first.
5.  Paste the second block into another new cell and run it. Does
    scikit-learn’s one-question tree ask the same question?

**The first block:**

``` python
column = "OverallQual"          # then the other two
cuts = np.unique(home_sales[column])[:-1]
left = [homes_left_over(column, cut) for cut in cuts]
best = int(np.argmin(____))
share = 100 * (1 - left[best] / home_spread)
print(f"{column} ≤ {cuts[best]}?  explains {share:.0f}%")
```

**The second block:**

``` python
home_stump = DecisionTreeRegressor(max_depth=1)
home_stump.fit(home_sales[home_columns], home_log_price)
print(describe_question(home_stump, 0))
```

## Exercise: Where the Models Go Wrong

- A score is an average over 1,460 homes. Where does each model miss
  most?

1.  Run the cell below: `biggest_misses(name)` lists a model’s ten worst
    homes. Its example: the one tree’s three worst.
2.  Click **+ Code** and run `biggest_misses("linear regression")`. In
    another new cell, run `biggest_misses("boosted trees")`.
3.  Read **miss %** (+100 is twice the price): which model misses worst?
4.  Read the **sale** column: what do the big misses have in common?

In [ ]:
# Each model's biggest misses, out of fold: the model never saw these homes.
home_price = np.exp(log_price)                          # back from log price to dollars
normal_sales = homes["SaleCondition"] == "Normal"       # an ordinary sale: 4 homes in 5


def biggest_misses(name, how_many=10):
    """The homes this model forecast worst: the price, the forecast and the miss (forecast / price
    - 1, in percent: +100 is twice the price), with the home's living area and kind of sale."""
    forecast = pd.Series(np.exp(homes_forecasts[name]), index=homes.index).round(-3)
    miss = 100 * (forecast / home_price - 1)
    worst = miss.abs().sort_values(ascending=False).index[:how_many]
    table = pd.DataFrame({"price": home_price[worst], "forecast": forecast[worst],
                          "miss %": miss[worst], "living area": homes.loc[worst, "GrLivArea"],
                          "sale": homes.loc[worst, "SaleCondition"]})
    print(table.to_string(formatters={
        "price": "{:,.0f}".format, "forecast": "{:,.0f}".format, "miss %": "{:+.0f}".format,
        "living area": "{:,}".format}))


def score_on(rows, name):
    """R², %, of a model's out-of-fold forecasts, counting only the homes in rows."""
    actual, forecast = log_price[rows], homes_forecasts[name][rows.to_numpy()]
    return 100 * (1 - np.sum((actual - forecast) ** 2) / np.sum((actual - actual.mean()) ** 2))


biggest_misses("one tree, depth 4", 3)

## Exercise: Set Aside the Odd Sales

- Most homes, 4 in 5, are ordinary sales: `SaleCondition == "Normal"`.
  **A choice:** score only those.

1.  Click **+ Code**, paste the code below and run it. Each model gets
    two scores: on all homes, and on the ordinary sales only. The models
    are the same; only the homes scored change.
2.  In each column, subtract the line’s score from boosting’s. Does the
    gap shrink when the odd sales are set aside?
3.  Rank the five models in each column. Does the order change?

``` python
print("                   all homes   normal")
for name in homes_forecasts:
    normal = score_on(normal_sales, name)
    score = homes_score[name]
    print(f"{name:18s}{score:10.1f}{normal:9.1f}")
```

## Exercise: A Deeper Tree?

- One tree finished last in the race. Is a deeper tree enough?

1.  Click **+ Code** and paste the code from the notebook, below these
    steps. It needs the race cell run first: the race builds
    `for_trees`, `folds` and `homes_r2`.
2.  Replace `____` with `None`: no depth limit. Run it.
3.  **new** scores homes each tree did not see; **fitted** scores the
    homes it was fit to. Does any depth catch linear regression’s score
    in the race?
4.  As the tree gets deeper, how does the gap between fitted and new
    change?
5.  Done early? Delete `min_samples_leaf=10` and run it again.

``` python
for depth in [2, 4, 8, ____]:
    tree = DecisionTreeRegressor(
        max_depth=depth, min_samples_leaf=10, random_state=0)
    one_tree = make_pipeline(for_trees, tree)
    forecasts = cross_val_predict(one_tree, homes, log_price,
                                  cv=folds)
    fitted = one_tree.fit(homes, log_price).predict(homes)
    print(f"depth {depth}: new {homes_r2(forecasts):.1f}%,"
          f" fitted {homes_r2(fitted):.1f}%")
```

## Few Columns, Little Gain

In [ ]:
# The same race with only three columns: quality, living area, year built.
three = ["OverallQual", "GrLivArea", "YearBuilt"]
three_score = {}
for name, racer in [("linear regression", make_pipeline(SimpleImputer(strategy="median"),
                                                          LinearRegression())),
                    ("boosted trees", make_pipeline(SimpleImputer(strategy="median"),
                                                    GradientBoostingRegressor(
                                                        n_estimators=300, learning_rate=0.05,
                                                        max_depth=3, random_state=0)))]:
    three_score[name] = homes_r2(cross_val_predict(racer, homes[three], log_price, cv=folds))

positions = np.arange(2)
plt.figure(figsize=(10, 2.4))
plt.bar(positions - 0.2, [three_score["linear regression"], homes_score["linear regression"]],
        width=0.4, color="gray", label="linear regression")
plt.bar(positions + 0.2, [three_score["boosted trees"], homes_score["boosted trees"]],
        width=0.4, color="tab:orange", label="boosted trees")
plt.xticks(positions, ["3 columns", "all 79"])
plt.ylim(60, 106)
plt.ylabel("R² (%)")
plt.legend(loc="upper left", ncol=2, frameon=False, fontsize=16)
plt.show()

- With 3 columns, boosting adds little over the line. With all 79, it
  adds much more.
- Trees gain from **many columns to combine**. One market with 12
  predictors, as in the equity premium, gives them little to work with.

# The Lab: Truth Known

## Predict: Who Comes Closest?

- The housing example shows that boosting can work. Now return to our
  simulated stocks, where we know the truth and can see why.
- The world of “A World We Control”: value pays only among small firms;
  momentum pays a little everywhere; three noise columns.
- Each model learns from the 6,000 fitting rows and is scored on the
  20,000 new ones.
- **Predict first:** which comes closest to the best possible?
  1.  OLS on all six characteristics
  2.  One tree, depth 2
  3.  Boosted trees, depth 2
  4.  Boosted stumps, depth 1

## The Horse Race

In [ ]:
models = {
    "OLS, all six": LinearRegression(),
    "one tree, depth 2": DecisionTreeRegressor(max_depth=2),
    "boosted stumps": GradientBoostingRegressor(max_depth=1, random_state=0),
    "boosted, depth 2": GradientBoostingRegressor(max_depth=2, random_state=0),
}
race = {"best possible": best_possible}
for name, model in models.items():
    model.fit(X_fit, y_fit)
    race[name] = new_data_r2(model.predict(X_new))
boosted_depth_2 = models["boosted, depth 2"]           # used again below

colors = ["gray", "tab:blue", "tab:blue", "tab:orange", "tab:orange"]
plt.figure(figsize=(10, 2.7))
plt.barh(list(race), list(race.values()), color=colors)
for position, score in enumerate(race.values()):
    plt.text(score + 0.1, position, f"{score:.2f}", va="center")
plt.gca().invert_yaxis()
plt.xlim(0, 7.8)
plt.xlabel("new-data R² (%)")
plt.show()

- Boosted trees of depth 2 come closest to the best possible.
- The others stay well below it; one tree with no depth limit is far
  below zero.

## Stumps Cannot See Interactions

- A stump asks one question, such as “value ≤ 0?”. Its step is the same
  for small and big firms.
- 100 stumps add up to one staircase per characteristic, added together:
  still “straight lines add up”, with steps.
- A depth-2 tree can ask about size **under** a question about value:
  “only among small firms”.
- So the depth sets which interactions boosting can find. Read: ESL
  10.11.

## Predict: More and More Trees

- Boosted trees of depth 2, learning rate 0.1, the same world.
- Add trees, from 1 to 1,000.
- **Predict first:** what happens to the fit and the new-data $R^2$?
  1.  Both keep rising
  2.  The fit keeps rising; new data rises, then falls
  3.  Both rise, then fall
  4.  New data falls from the first tree on

## The Number of Trees Is a Dial

In [ ]:
many = GradientBoostingRegressor(max_depth=2, n_estimators=1000, random_state=0)
many.fit(X_fit, y_fit)
# staged_predict hands out the forecasts after 1 tree, 2 trees, ..., 1,000 trees: one fit.
fit_by_count = [fitted_r2(forecasts) for forecasts in many.staged_predict(X_fit)]
new_by_count = [new_data_r2(forecasts) for forecasts in many.staged_predict(X_new)]
counts = np.arange(1, 1001)

plt.figure(figsize=(10, 2.8))
plt.plot(counts, fit_by_count, color="tab:blue", linewidth=3)
plt.plot(counts, new_by_count, color="tab:orange", linewidth=3)
plt.axhline(best_possible, color="gray", linestyle="--")
plt.axhline(0, color="black", linewidth=0.8)
plt.text(1150, fit_by_count[-1], "fit", color="tab:blue", va="center")
plt.text(1150, new_by_count[-1] + 1.5, "new data", color="tab:orange", va="center")
plt.text(1.2, best_possible + 1.5, "best possible", color="gray")
plt.xscale("log")
plt.xlim(1, 6000)
plt.ylim(-2, fit_by_count[-1] + 4)
plt.xticks([1, 10, 100, 1000], ["1", "10", "100", "1,000"])
plt.xlabel("number of trees (log scale)")
plt.ylabel("R² (%)")
plt.show()

- On new data, the score peaks after a few dozen trees, then falls.
- The fit rises all the way.

## Choose It on Validation Data

In [ ]:
# Hold out the last 1,500 fitting rows as a validation block; grow 1,000 trees on the rest.
cut = 4500
trial = GradientBoostingRegressor(max_depth=2, n_estimators=1000, random_state=0)
trial.fit(X_fit.iloc[:cut], y_fit.iloc[:cut])
validation_error = [np.mean((y_fit.iloc[cut:] - forecasts) ** 2)
                    for forecasts in trial.staged_predict(X_fit.iloc[cut:])]
chosen = int(np.argmin(validation_error)) + 1

# Refit with the chosen number of trees on all 6,000 rows, then score on new data.
honest = GradientBoostingRegressor(max_depth=2, n_estimators=chosen, random_state=0)
honest.fit(X_fit, y_fit)
honest_r2 = new_data_r2(honest.predict(X_new))

plt.figure(figsize=(10, 2.6))
plt.plot(counts, 100 * np.array(validation_error), color="tab:blue", linewidth=3)
plt.axvline(chosen, color="tab:orange", linestyle="--", linewidth=2)
plt.text(chosen * 1.15, 100 * max(validation_error), f"chosen: {chosen}", color="tab:orange",
         va="top")
plt.xscale("log")
plt.xticks([1, 10, 100, 1000], ["1", "10", "100", "1,000"])
plt.xlabel("number of trees (log scale)")
plt.ylabel("validation error\n(× 100)")
plt.show()

- The validation block picks a number of trees; refit on all 6,000 rows
  and score on new data.
- It lands close to the hindsight choice, and it is honest.

## Your Turn: Change the Depth

- Boosted trees with scikit-learn’s other defaults (learning rate 0.1,
  100 trees). Try depth 1, 2 and 3.

``` python
boosted = GradientBoostingRegressor(max_depth=____,
                                    random_state=0)
boosted.fit(fitting[CHARACTERISTICS], fitting["return"])
forecasts = boosted.predict(new[CHARACTERISTICS])
print(f"new-data R²: {new_data_r2(forecasts):.2f}%")
```

## Check: Depth for an Interaction

- Value pays only among small firms.
- **Fingers up:** what is the smallest tree depth that lets boosting
  capture that?
  1.  Depth 1
  2.  Depth 2
  3.  Depth 3
  4.  Any depth: boosting adds enough trees

# Opening the Box

## Predict: Where Does Pure Noise Rank?

- Rank the six characteristics by how much the boosted trees rely on
  them: size, value, momentum and three columns of pure noise.
- **Predict first:** where do the noise columns land?
  1.  At the bottom, near zero
  2.  At the top: trees use everything
  3.  In the middle
  4.  It cannot be measured

## Shuffle One Column

- Take the new data and **shuffle one column**: its values, in random
  order. Every other column stays as it was.
- That column’s link to returns is broken. Forecast again.
- The new-data $R^2$ lost is the column’s **importance**.
- It works for any model: OLS, trees, networks.

## Which Characteristics Matter?

In [ ]:
# Shuffle each column of the new data 5 times; record the R² lost (scikit-learn does it all).
shuffled = permutation_importance(boosted_depth_2, X_new, new["return"], scoring="r2",
                                  n_repeats=5, random_state=0)
importance = pd.Series(100 * shuffled.importances_mean, index=CHARACTERISTICS)

plt.figure(figsize=(10, 2.7))
plt.barh(CHARACTERISTICS, importance, color="tab:blue")
for position, points in enumerate(importance):
    plt.text(max(points, 0) + 0.05, position, f"{points:.2f}".replace("-", "−"), va="center")
plt.gca().invert_yaxis()
plt.xlim(-0.2, importance.max() * 1.25)
plt.xlabel("new-data R² lost when the column is shuffled (points)")
plt.show()

- Value first. Then size and momentum: size matters only through value,
  and still counts.
- The noise columns: about zero.

## Partial Dependence

In [ ]:
# Set every stock's value to the same level, forecast, and average: once for small firms,
# once for big ones. Repeat for each level of value from -0.9 to 0.9.
levels = np.linspace(-0.9, 0.9, 37)
dependence = {}
plt.figure(figsize=(10, 1.8))
for label, color, firms in [("small firms", "tab:orange", new[new["size"] < 0]),
                            ("big firms", "tab:blue", new[new["size"] > 0])]:
    average_forecasts = []
    for level in levels:
        moved = firms[CHARACTERISTICS].copy()
        moved["value"] = level
        average_forecasts.append(100 * boosted_depth_2.predict(moved).mean())
    dependence[label] = average_forecasts
    plt.plot(levels, average_forecasts, color=color, linewidth=3, label=label)
plt.plot(levels, 100 * 0.06 * levels, color="tab:orange", linestyle="--", linewidth=1.5)
plt.plot(levels, 0 * levels, color="tab:blue", linestyle="--", linewidth=1.5)
plt.xlabel("value (everyone set to this level)")
plt.ylabel("average\nforecast (%)")
plt.legend(loc="upper left", frameon=False)
plt.show()

- Change the value characteristic, leave the other inputs unchanged, and
  average the model’s forecasts, separately for small and big firms.
- Small firms: a steep staircase, near the truth (dashed). Big firms:
  nearly flat. The model learned “only among small firms”.

## Check: Reading Partial Dependence

- A partial-dependence plot for value: the line rises steeply for small
  firms and is flat for big firms.
- **Fingers up:** what does it say?
  1.  Value matters only among small firms
  2.  Value does not matter, on average
  3.  Size matters more than value
  4.  Small firms earn more than big ones

# Predicting Bankruptcy

## Yes or No: The Same Rule

- Code failed as **1**, survived as **0**. A leaf’s average is then its
  **share** of failures, p: its forecast.
- In a leaf, the squared gaps from p add up to firms × p × (1 − p).
- The tree asks the question whose leaves leave the least: the
  **purest** leaves. scikit-learn’s classifier uses 2 × p × (1 − p), the
  **Gini** index: the same choice.

## The Data: Firms and Ratios

In [ ]:
# Polish manufacturing firms (Zieba, Tomczak and Tomczak, 2016; UCI Machine Learning
# Repository, CC BY 4.0). One row is one firm: 64 financial ratios from its accounts, and
# whether it went bankrupt within `horizon` years. Many firms appear at more than one horizon.
firms = pd.read_csv(resolve_data_path("polish_bankruptcy.csv.gz"))
ratio_columns = [f"attr{number}" for number in range(1, 65)]
by_horizon = firms.groupby("horizon")["bankrupt"].agg(firms="size", failed="sum")
by_horizon["failed %"] = (100 * by_horizon["failed"] / by_horizon["firms"]).round(1)
by_horizon.index.name = "years ahead"
print(by_horizon.sort_index(ascending=False))
print(f"share of ratios missing: {100 * firms[ratio_columns].isna().mean().mean():.1f}%")

- **One row** is one Polish manufacturing firm: 64 ratios from its
  accounts (profit, debt, liquidity, turnover).
- **Target:** did it go bankrupt within 1 to 5 years? Only about one
  firm in twenty does.
- Many firms appear at more than one horizon.

## Accuracy Misleads

- **Only about one firm in twenty fails.** A “model” that says “nobody
  fails” is right about 95% of the time, and finds no failing firm.
- So we **rank** firms by risk and ask a different question.
- **AUC:** pick one failing and one healthy firm at random. What is the
  chance the model gives the failing one the higher risk score?
  - 0.5: a coin flip. 1.0: always right.

## Predict: Who Ranks Best?

- Four models, 64 ratios, five horizons, scored by AUC on firms not used
  for fitting.
- **Predict first:** which has the highest AUC at every horizon?
  1.  Logit: the simplest
  2.  One tree, depth 4
  3.  Random forest
  4.  Boosted trees

## Four Models, Five Horizons

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


def bankruptcy_scores(horizon, seed):
    """Fit four models on a random 70% of one horizon's firms. Return the other 30%'s outcomes
    and each model's risk score for them (a higher score means more likely to fail)."""
    table = firms[firms["horizon"] == horizon]
    fit, test = train_test_split(table, test_size=0.3, stratify=table["bankrupt"],
                                 random_state=seed)
    to_median = SimpleImputer(strategy="median")      # fill a blank ratio with the median
    models = {
        "logit": make_pipeline(to_median, StandardScaler(),
                               LogisticRegression(C=0.1, max_iter=2000)),
        "one tree": make_pipeline(to_median, DecisionTreeClassifier(
            max_depth=4, min_samples_leaf=30, random_state=0)),
        "forest": make_pipeline(to_median, RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=0)),
        "boosting": HistGradientBoostingClassifier(     # takes blank ratios as they are
            max_depth=3, learning_rate=0.05, max_iter=300, random_state=0),
    }
    scores = {}
    for name, model in models.items():
        model.fit(fit[ratio_columns], fit["bankrupt"])
        scores[name] = model.predict_proba(test[ratio_columns])[:, 1]
    return test["bankrupt"].to_numpy(), scores

Run one horizon, one split (a few seconds). The slide’s figure repeats
it for every horizon and five random splits.

In [ ]:
outcomes, scores = bankruptcy_scores(horizon=3, seed=0)
for name, risk in scores.items():
    print(f"{name:9s} AUC {roc_auc_score(outcomes, risk):.3f}")

- Boosting ranks best at every horizon. Then the forest, then one tree;
  the logit is last or tied.
- Each point averages five random splits. More years ahead is not harder
  here.

## Check: Reading an AUC

- A model’s AUC on a set of firms is 0.5.
- **Fingers up:** what does it do?
  1.  Ranks firms as well as a coin flip
  2.  Ranks every failing firm above every healthy one
  3.  Is right half the time
  4.  Flags half of the failing firms

## Back to the Equity Premium

- Week 2’s market premium: 12 predictors, 672 monthly forecasts, boosted
  trees tuned honestly. Every version loses to the historical mean, as
  the kitchen-sink regression did.
- The trees’ forecasts swing far more than the historical mean’s. Mostly
  noise.
- **A thin set of features:** one series and 12 predictors, against
  thousands of firms and dozens of columns above. Trees need both.

## Nonlinearity Needs Data

- The same world with a realistic signal: the best possible $R^2$ is
  below 1%, as for real stocks.
- With few rows, boosted trees lose to OLS; with hundreds of thousands,
  they come close to the best possible.

## Where Trees Win: Thousands of Stocks

- **Gu, Kelly & Xiu (2020), their numbers:** almost 30,000 US stocks,
  1957–2016, 94 characteristics; forecasts tested on 1987–2016.
- Monthly out-of-sample $R^2$, in %:

| Model                         | Each stock | S&P 500, from the stock forecasts |
|-------------------------------|------------|-----------------------------------|
| OLS, size, value and momentum | 0.16       | −0.22                             |
| Random forest                 | 0.33       | 1.37                              |
| Boosted trees                 | 0.34       | 1.40                              |
| Neural network, 3 layers      | 0.40       | 1.80                              |

- OLS on all 920 of their predictors: −3.46% per stock.
- Their reading: the gain comes from **interactions**. Curves in each
  characteristic alone did no better than a straight line.

## Check: When Do Trees Win?

- **Fingers up:** when do boosted trees beat a straight line out of
  sample?
  1.  Few observations, a strong signal, no interactions
  2.  Many observations, and interactions or thresholds in the truth
  3.  Always: trees are more flexible
  4.  Small samples, because trees ignore outliers

# Random Forests, Briefly

## Average Many Deep Trees

In [ ]:
# 200 trees, each grown on a resampled copy of the rows (bootstrap), with no depth limit; at
# each question a tree may use only 2 of the 6 characteristics, picked at random.
forests = {
    "forest, leaves of 1 stock": RandomForestRegressor(n_estimators=200, max_features=2,
                                                      random_state=0, n_jobs=-1),
    "forest, leaves of 100+": RandomForestRegressor(n_estimators=200, max_features=2,
                                                   min_samples_leaf=100, random_state=0,
                                                   n_jobs=-1),
}
for name, forest in forests.items():
    forest.fit(X_fit, y_fit)
    print(f"{name:26s} new-data R² {new_data_r2(forest.predict(X_new)):5.2f}%")
print(f"{'boosted, depth 2':26s} new-data R² {race['boosted, depth 2']:5.2f}%")

- Each tree is deep and noisy; the average is much calmer.
- On noisy returns even a forest needs big leaves. Then it ties with
  boosting.

## Boosting Against Forests

|  | Boosting | Random forest |
|------------------------|------------------------|------------------------|
| The trees | small: depth 1 to 3 | deep |
| Built | one after another | side by side |
| Each tree | fixes what the others still get wrong | sees a resampled copy of the rows |
| The forecast | the sum of small steps | the average |
| It cuts | bias: one small tree is too simple | variance: one deep tree is too noisy |
| Main dials | depth, learning rate, number of trees | leaf size, predictors per question |

- In GKX’s stocks and in our world, the two finish close. Read: ISLP
  8.2; ESL 15.1–15.3.

## Check: Forest or Boosting?

- **Fingers up:** which describes a random forest?
  1.  Small trees fitted one after another to the errors
  2.  Deep trees fitted on resampled data, then averaged
  3.  One deep tree, pruned back
  4.  A straight line with a penalty

# Neural Networks, Briefly

## One Neuron: Add, Then Bend

In [ ]:
# One neuron: a weighted sum of the characteristics (add), then max(0, z) (bend). The weights
# are made up, to show the arithmetic; a network learns them from data.
weights = {"size": -0.4, "value": 0.6, "momentum": 0.5}
intercept = 0.1
stocks = {"stock A": {"size": -0.5, "value": 0.8, "momentum": 0.2},
          "stock B": {"size": 0.5, "value": -0.8, "momentum": -0.2}}
sums = {name: intercept + sum(weights[key] * value for key, value in row.items())
        for name, row in stocks.items()}              # the "add" step
outputs = {name: max(0, z) for name, z in sums.items()}   # the "bend" step

z_grid = np.linspace(-1.2, 1.2, 100)
plt.figure(figsize=(10, 2.3))
plt.plot(z_grid, np.maximum(0, z_grid), color="tab:orange", linewidth=3)
for name, z in sums.items():
    plt.scatter(z, outputs[name], s=120, color="black", zorder=3)
    plt.text(z, outputs[name] + 0.2, name, ha="center")
plt.xlabel("z: the weighted sum")
plt.ylabel("neuron output")
for side in ["top", "right"]:
    plt.gca().spines[side].set_visible(False)
plt.show()

- **Add:**
  $z = 0.1 - 0.4\,\text{size} + 0.6\,\text{value} + 0.5\,\text{momentum}$.
- **Bend:** output $= \max(0, z)$: a negative sum is cut to zero, and
  the neuron is off.

## A Network Is Stacked Regressions

- Each hidden neuron: a regression on the inputs, then $\max(0, \cdot)$,
  a hockey stick. The forecast: a regression on the neurons.
- No hidden layer: a plain linear regression. Read: ISLP 10.1–10.2.

## Hockey Sticks Add Up to Curves

In [ ]:
# The curve of "Boosting, Tree by Tree". Four hockey sticks, max(0, x - bend), with bends
# fixed at -1, -0.5, 0 and 0.5; OLS chooses how much of each to add up.
bends = [-1.0, -0.5, 0.0, 0.5]
sticks = np.column_stack([np.maximum(0, x_curve - bend) for bend in bends])
combination = LinearRegression().fit(sticks, y_curve)

figure, axes = plt.subplots(1, 2, figsize=(11, 2.7))
for k, bend in enumerate(bends):
    axes[0].plot(x_curve, sticks[:, k], linewidth=2.5,
                 label=f"bends at {bend:g}".replace("-", "−"))
axes[0].set_title("four hockey sticks", fontsize=17)
axes[0].legend(loc="upper left", frameon=False, fontsize=14)
axes[1].set_title("added up, weights from OLS", fontsize=17)
axes[1].scatter(x_curve, y_curve, color="lightgray", s=10)
axes[1].plot(x_curve, truth_curve, color="black", linestyle="--", linewidth=2)
axes[1].plot(x_curve, snapshots[100], color="tab:orange", linewidth=2.5, label="100 stumps")
axes[1].plot(x_curve, combination.predict(sticks), color="tab:blue", linewidth=3,
             label="sticks added up")
axes[1].legend(loc="lower right", frameon=False, fontsize=15)
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_xlabel("x")
plt.subplots_adjust(wspace=0.08)
plt.show()

- Four hockey sticks, added up, bend into the curve; trees use steps.
- A network also learns where each stick bends. Here we fixed the bends.

## What GKX Found

- **Their numbers:** a network with one hidden layer scored 0.33% a
  month per stock; with three layers 0.40%, their best.
- Four and five layers did no better. Their boosted trees and forests
  used small trees: fewer than six leaves on average.
- “Shallow learning outperforms deeper learning”: too little data and a
  tiny signal, as they put it, compared with computer vision.
- Networks and boosted trees finish close; networks need more choices.

## The Costs

In [ ]:
# Two small networks on the simulated world, each started from three random places.
# early_stopping: hold out 10% of the rows and stop training when they stop improving.
network_r2 = {}
for layers in [(8,), (32, 16)]:
    network_r2[layers] = []
    for start in [0, 1, 2]:
        network = MLPRegressor(hidden_layer_sizes=layers, early_stopping=True, max_iter=500,
                               random_state=start)
        network.fit(X_fit, y_fit)
        network_r2[layers].append(new_data_r2(network.predict(X_new)))
    scores = ", ".join(f"{score:.2f}" for score in network_r2[layers])
    print(f"neurons {str(layers):9s} three starts: {scores}%")
print(f"boosted trees, depth 2: {race['boosted, depth 2']:.2f}%")

- Many dials: layers, neurons, learning rate, how long to train.
- A different random start gives a different answer. For returns,
  boosted trees give most of the gain with fewer choices.

## Check: No Hidden Layer

- A network with no hidden layer and no hockey sticks.
- **Fingers up:** it is the same as:
  1.  A linear regression
  2.  A tree of depth 1
  3.  A random forest
  4.  The historical mean

# Close

## Week 4 in One Slide

- A straight line adds effects up. Real returns have an interaction:
  value pays among small firms.
- A tree asks yes-or-no questions and forecasts leaf averages. Grown
  deep, it memorizes.
- Boosting adds many small trees, each fixing what the others still get
  wrong. Tune depth, learning rate and the number of trees on validation
  data.
- Trees win when there is something to find and enough data: thousands
  of stocks, not 672 months of one market.

## Not in This Deck

- **XGBoost** and **LightGBM:** faster boosting, the same idea; the
  usual choice in practice.
- **SHAP values:** another way to open the box, one forecast at a time.
- **Deep networks** for images and text; text comes in the text deck.

## Readings

**Meeting 7: trees and boosting.**

- Required: Hastie, Tibshirani & Friedman, *The Elements of Statistical
  Learning*, sections 9.2 and 10.9–10.13.
- Gentler: James, Witten, Hastie, Tibshirani & Taylor (2023), *ISLP*,
  8.1–8.2.
- Optional: *The Elements of Statistical Learning* 15.1–15.3 (random
  forests).

**Meeting 8: machine learning for returns.**

- Required: Gu, Kelly & Xiu (2020), “Empirical Asset Pricing via Machine
  Learning”, *Review of Financial Studies* 33(5), 2223–2274.
- Neural networks: *The Elements of Statistical Learning* 11.3–11.5;
  gentler, *ISLP* 10.1–10.2.